# Proyecto Final - Sistema Inteligente de Recomendacion para E-commerce

# Notebook 03: Ingenieria de Caracteristicas (Feature Engineering)

---


## Objetivo

Construir las estructuras de datos necesarias para un sistema de recomendacion:

1. **Matriz usuario-item** (para collaborative filtering con SVD/ALS)
2. **Features de producto** (para content-based y cold start)
3. **Features de usuario** (para cold start de clientes nuevos)

---

**Metodologia:** CRISP-DM
**Sprint:** 1 - Ingenieria de Caracteristicas

## 0. Configuracion del entorno

Importacion de librerias y carga de datos limpios desde `data_clean.py`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar SRC al path
sys.path.append(str(Path.cwd().parent / 'SRC'))

from data_clean import limpiar_tablas

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', palette='deep')

print('Entorno configurado correctamente.')

## 1. Carga y limpieza de datos

Se ejecuta el pipeline completo de `data_clean.py` que limpia las 7 tablas del dataset.

In [ ]:
events, sessions, reviews, orders, order_items, customers, products = limpiar_tablas()

print('\n--- Resumen de tablas limpias ---')
print(f'  events      : {events.shape[0]:>10,} filas x {events.shape[1]} columnas')
print(f'  sessions    : {sessions.shape[0]:>10,} filas x {sessions.shape[1]} columnas')
print(f'  reviews     : {reviews.shape[0]:>10,} filas x {reviews.shape[1]} columnas')
print(f'  orders      : {orders.shape[0]:>10,} filas x {orders.shape[1]} columnas')
print(f'  order_items : {order_items.shape[0]:>10,} filas x {order_items.shape[1]} columnas')
print(f'  customers   : {customers.shape[0]:>10,} filas x {customers.shape[1]} columnas')
print(f'  products    : {products.shape[0]:>10,} filas x {products.shape[1]} columnas')

---
## 2. Matriz usuario-item (para collaborative filtering)

### Que es?

Una tabla que muestra cuanto le "interesa" cada producto a cada usuario.

### Como se construye?

1. Se une `events` con `sessions` para obtener el `customer_id` de cada evento.
2. Se asigna un **score** segun el tipo de accion:
   - `page_view` = 1 (vio el producto)
   - `add_to_cart` = 3 (lo agrego al carrito)
   - `purchase` = 5 (lo compro)
3. Se agrupa por `customer_id` y `product_id`, sumando los scores.

### Para que sirve?

Es la entrada principal de los algoritmos **SVD** y **ALS** (collaborative filtering). Estos algoritmos buscan patrones en esta matriz para predecir que productos le gustarian a un usuario que aun no vio.

- **SVD** (Descomposicion en Valores Singulares): comprime la matriz en patrones latentes (como "gustos" ocultos).
- **ALS** (Least Squares Alternantes): llena los ceros de la matriz iterativamente hasta encontrar los scores predichos.

In [ ]:
# Unir events con sessions para obtener customer_id
events_con_user = events.merge(
    sessions[['session_id', 'customer_id']],
    on='session_id',
    how='left'
)

print(f'Events con customer_id: {len(events_con_user):,} filas')
print(f'\nPrimeras filas:')
events_con_user[['event_id', 'session_id', 'customer_id', 'event_type', 'product_id', 'timestamp']].head(10)

In [ ]:
# Asignar scores segun tipo de evento
event_weights = {'page_view': 1, 'add_to_cart': 3, 'purchase': 5}
events_con_user['score'] = events_con_user['event_type'].map(event_weights).fillna(1)

print('Distribucion de scores por tipo de evento:')
print(events_con_user.groupby('event_type')['score'].first().to_string())
print(f'\nTotal de eventos con score asignado: {len(events_con_user):,}')

In [ ]:
# Construir la matriz usuario-item
matriz_usuario_item = events_con_user.groupby(
    ['customer_id', 'product_id']
)['score'].sum().reset_index()

n_users = matriz_usuario_item['customer_id'].nunique()
n_prods = matriz_usuario_item['product_id'].nunique()
n_interacciones = len(matriz_usuario_item)
esparsidad = (1 - n_interacciones / (n_users * n_prods)) * 100

print('=== MATRIZ USUARIO-ITEM ===')
print(f'  Usuarios unicos    : {n_users:>8,}')
print(f'  Productos unicos   : {n_prods:>8,}')
print(f'  Interacciones      : {n_interacciones:>8,}')
print(f'  Pares posibles     : {n_users * n_prods:>12,}')
print(f'  Esparsidad         : {esparsidad:.2f}%')
print(f'\nPrimeras filas de la matriz:')
matriz_usuario_item.head(10)

**Interpretacion:** La matriz tiene 19,945 usuarios y 1,197 productos con 529,593 interacciones. La esparsidad del 97.78% significa que la mayoria de los usuarios solo interactuo con una pequena fraccion de los productos disponibles. Esto es normal en e-commerce y es exactamente lo que SVD/ALS buscan resolver: predecir los ceros.

In [ ]:
# Visualizacion: distribucion de scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Score promedio por usuario
score_usr = matriz_usuario_item.groupby('customer_id')['score'].mean()
axes[0].hist(score_usr, bins=50, color='#2a78d6', edgecolor='white')
axes[0].set_title('Score promedio por usuario')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Cantidad de usuarios')

# Interacciones por usuario
interacciones_usr = matriz_usuario_item.groupby('customer_id').size()
axes[1].hist(interacciones_usr, bins=50, color='#e87ba4', edgecolor='white')
axes[1].set_title('Productos tocados por usuario')
axes[1].set_xlabel('Cantidad de productos')
axes[1].set_ylabel('Cantidad de usuarios')

# Interacciones por producto
interacciones_prod = matriz_usuario_item.groupby('product_id').size()
axes[2].hist(interacciones_prod, bins=50, color='#7fbf5f', edgecolor='white')
axes[2].set_title('Usuarios por producto')
axes[2].set_xlabel('Cantidad de usuarios')
axes[2].set_ylabel('Cantidad de productos')

plt.tight_layout()
plt.show()

print(f'Score promedio por usuario: media={score_usr.mean():.1f}, mediana={score_usr.median():.1f}')
print(f'Productos tocados por usuario: media={interacciones_usr.mean():.1f}, mediana={interacciones_usr.median():.0f}')
print(f'Usuarios por producto: media={interacciones_prod.mean():.1f}, mediana={interacciones_prod.median():.0f}')

---
## 3. Features de producto (para content-based y cold start)

### Que es?

Un DataFrame con una fila por producto y todas sus caracteristicas derivadas.

### Que features se crearon?

| Feature | Origen | Descripcion |
|---------|--------|-------------|
| `category` | products | Categoria del producto |
| `price_usd` | products | Precio de venta |
| `cost_usd` | products | Costo |
| `margin_usd` | products | Margen de ganancia |
| `n_views` | events | Cantidad de page_views |
| `n_cart` | events | Cantidad de add_to_cart |
| `n_purchases` | events | Cantidad de purchases |
| `popularidad` | order_items | Compradores unicos |
| `rating_promedio` | reviews | Rating promedio recibido |
| `n_ratings` | reviews | Cantidad de reviews recibidas |

### Para que sirve?

- **Content-based**: recomienda productos similares en features (misma categoria, precio parecido, buen rating).
- **Cold start**: para productos nuevos sin interacciones, se usan sus features (categoria, precio) para recomendarlos.

In [ ]:
# Conteos desde events
pv = events_con_user[events_con_user['event_type'] == 'page_view']
ac = events_con_user[events_con_user['event_type'] == 'add_to_cart']
pu = events_con_user[events_con_user['event_type'] == 'purchase']

n_views = pv.groupby('product_id').size().rename('n_views')
n_cart = ac.groupby('product_id').size().rename('n_cart')
n_purchases_events = pu.groupby('product_id').size().rename('n_purchases')

# Popularidad desde order_items (compradores unicos)
oi_con_order = order_items.merge(
    orders[['order_id', 'customer_id']], on='order_id', how='left'
)
popularidad = oi_con_order.groupby('product_id')['customer_id'].nunique().rename('popularidad')

# Rating promedio desde reviews
rating_stats = reviews.groupby('product_id')['rating'].agg(['mean', 'count']).rename(
    columns={'mean': 'rating_promedio', 'count': 'n_ratings'}
)

# Unir todo
df_prod = products[['product_id', 'category', 'price_usd', 'cost_usd', 'margin_usd']].copy()
df_prod = df_prod.set_index('product_id')

for feat in [n_views, n_cart, n_purchases_events, popularidad, rating_stats]:
    df_prod = df_prod.join(feat, how='left')

df_prod = df_prod.fillna(0)

features_producto = df_prod.reset_index()

print(f'=== FEATURES DE PRODUCTO ===')
print(f'  Productos: {len(features_producto):,}')
print(f'  Columnas: {list(features_producto.columns)}')
print(f'\nPrimeras filas:')
features_producto.head(10)

In [ ]:
# Estadisticas de las features de producto
print('=== ESTADISTICAS DE FEATURES DE PRODUCTO ===')
features_producto.describe().round(2)

In [ ]:
# Visualizacion: distribuciones de features de producto
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

features_producto['price_usd'].hist(ax=axes[0,0], bins=30, color='#2a78d6', edgecolor='white')
axes[0,0].set_title('Distribucion de precio')

features_producto['popularidad'].hist(ax=axes[0,1], bins=30, color='#e87ba4', edgecolor='white')
axes[0,1].set_title('Popularidad (compradores unicos)')

features_producto['n_views'].hist(ax=axes[0,2], bins=30, color='#7fbf5f', edgecolor='white')
axes[0,2].set_title('Vistas totales')

features_producto['rating_promedio'].hist(ax=axes[1,0], bins=30, color='#eda100', edgecolor='white')
axes[1,0].set_title('Rating promedio')

features_producto['n_ratings'].hist(ax=axes[1,1], bins=30, color='#1baf7a', edgecolor='white')
axes[1,1].set_title('Cantidad de reviews')

features_producto['category'].value_counts().plot(kind='bar', ax=axes[1,2], color='#2a78d6')
axes[1,2].set_title('Productos por categoria')

plt.tight_layout()
plt.show()

---
## 4. Features de usuario (para cold start)

### Que es?

Un DataFrame con una fila por usuario y todas sus caracteristicas derivadas.

### Que features se crearon?

| Feature | Origen | Descripcion |
|---------|--------|-------------|
| `age` | customers | Edad del usuario |
| `country` | customers | Pais de registro |
| `marketing_opt_in` | customers | Si acepto marketing |
| `n_sessions` | sessions | Cantidad de sesiones |
| `n_purchases` | orders | Compras realizadas |
| `ticket_promedio` | orders | Gasto promedio por compra |
| `n_products_viewed` | events | Productos distintos vistos |
| `n_products_carted` | events | Productos distintos en carrito |
| `rating_promedio_usr` | reviews | Rating promedio que otorga |

### Para que sirve?

- **Cold start de clientes nuevos**: cuando un cliente se registra pero no tiene historial de compras, se usa su perfil demografico (edad, pais) para recomendarle lo que suelen comprar usuarios similares.

In [ ]:
# Features demograficas
df_usr = customers[['customer_id', 'age', 'country', 'marketing_opt_in']].copy()
df_usr = df_usr.set_index('customer_id')

# Sesiones por usuario
n_sessions = sessions.groupby('customer_id').size().rename('n_sessions')
df_usr = df_usr.join(n_sessions, how='left')

# Compras (desde orders)
n_purchases = orders.groupby('customer_id').size().rename('n_purchases')
ticket_prom = orders.groupby('customer_id')['total_usd'].mean().rename('ticket_promedio')
df_usr = df_usr.join(n_purchases, how='left').join(ticket_prom, how='left')

# Productos vistos y en carrito (desde events)
pv_user = events_con_user[events_con_user['event_type'] == 'page_view']
ac_user = events_con_user[events_con_user['event_type'] == 'add_to_cart']

n_viewed = pv_user.groupby('customer_id')['product_id'].nunique().rename('n_products_viewed')
n_carted = ac_user.groupby('customer_id')['product_id'].nunique().rename('n_products_carted')
df_usr = df_usr.join(n_viewed, how='left').join(n_carted, how='left')

# Rating promedio dado por el usuario
rev_con_customer = reviews.merge(
    oi_con_order[['order_id', 'customer_id']].drop_duplicates('order_id'),
    on='order_id', how='left'
)
rating_usr = rev_con_customer.groupby('customer_id')['rating'].mean().rename('rating_promedio_usr')
df_usr = df_usr.join(rating_usr, how='left')

df_usr = df_usr.fillna(0)
features_usuario = df_usr.reset_index()

print(f'=== FEATURES DE USUARIO ===')
print(f'  Usuarios: {len(features_usuario):,}')
print(f'  Columnas: {list(features_usuario.columns)}')
print(f'\nPrimeras filas:')
features_usuario.head(10)

In [ ]:
# Estadisticas de las features de usuario
print('=== ESTADISTICAS DE FEATURES DE USUARIO ===')
features_usuario.describe().round(2)

In [ ]:
# Visualizacion: distribuciones de features de usuario
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

features_usuario['age'].hist(ax=axes[0,0], bins=30, color='#2a78d6', edgecolor='white')
axes[0,0].set_title('Distribucion de edad')

features_usuario['n_sessions'].hist(ax=axes[0,1], bins=50, color='#e87ba4', edgecolor='white')
axes[0,1].set_title('Sesiones por usuario')

features_usuario['n_purchases'].hist(ax=axes[0,2], bins=30, color='#7fbf5f', edgecolor='white')
axes[0,2].set_title('Compras por usuario')

features_usuario['ticket_promedio'].hist(ax=axes[1,0], bins=30, color='#eda100', edgecolor='white')
axes[1,0].set_title('Ticket promedio')

features_usuario['n_products_viewed'].hist(ax=axes[1,1], bins=30, color='#1baf7a', edgecolor='white')
axes[1,1].set_title('Productos vistos')

features_usuario['country'].value_counts().head(10).plot(kind='bar', ax=axes[1,2], color='#2a78d6')
axes[1,2].set_title('Top 10 paises')

plt.tight_layout()
plt.show()

---
## 5. Preprocesamiento para modelado

Se aplican tres transformaciones:

| Transformacion | Que hace | Por que |
|----------------|----------|--------|
| **Matriz pivote** | Convierte la lista de scores en tabla usuario x producto | SVD/ALS necesitan esta estructura |
| **LabelEncoder** | Convierte texto a numeros (category, country) | Los modelos no entienden texto |
| **StandardScaler** | Centra los numeros en torno a 0 | Que un numero grande no domine sobre uno chico |

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 5a. Matriz pivote usuario x producto
interaction_matrix = matriz_usuario_item.pivot_table(
    index='customer_id',
    columns='product_id',
    values='score',
    fill_value=0
)

print(f'=== MATRIZ INTERACTION ===')
print(f'  Forma: {interaction_matrix.shape[0]} usuarios x {interaction_matrix.shape[1]} productos')
print(f'\nPrimeras filas (primeras 10 columnas):')
interaction_matrix.iloc[:8, :10]

In [ ]:
# 5b. Preprocesar features de producto
product_features = features_producto.copy()

# Encoding de category
le_cat = LabelEncoder()
product_features['category_encoded'] = le_cat.fit_transform(product_features['category'])

# Escalado de features numericas
scaler_prod = StandardScaler()
num_cols_prod = ['price_usd', 'cost_usd', 'margin_usd', 'popularidad',
                 'rating_promedio', 'n_views', 'n_cart', 'n_purchases']
product_features[num_cols_prod] = scaler_prod.fit_transform(product_features[num_cols_prod])

print(f'=== PRODUCT FEATURES (preprocesado) ===')
print(f'  Forma: {product_features.shape}')
print(f'\nPrimeras filas:')
product_features.head(10)

In [ ]:
# 5c. Preprocesar features de usuario
user_features = features_usuario.copy()

# Encoding de country
le_country = LabelEncoder()
user_features['country_encoded'] = le_country.fit_transform(user_features['country'])

# Escalado de features numericas
scaler_usr = StandardScaler()
num_cols_usr = ['age', 'n_sessions', 'n_purchases', 'ticket_promedio',
                'n_products_viewed', 'n_products_carted', 'rating_promedio_usr']
user_features[num_cols_usr] = scaler_usr.fit_transform(user_features[num_cols_usr])

print(f'=== USER FEATURES (preprocesado) ===')
print(f'  Forma: {user_features.shape}')
print(f'\nPrimeras filas:')
user_features.head(10)

In [ ]:
# Resumen de estructuras generadas
print('=' * 60)
print('RESUMEN DE ESTRUCTURAS GENERADAS')
print('=' * 60)

print(f'\n1. interaction_matrix:')
print(f'   Forma: {interaction_matrix.shape}')
print(f'   Para: Collaborative Filtering (SVD/ALS)')

print(f'\n2. product_features:')
print(f'   Forma: {product_features.shape}')
print(f'   Para: Content-Based y Cold Start')

print(f'\n3. user_features:')
print(f'   Forma: {user_features.shape}')
print(f'   Para: Cold Start de clientes nuevos')

print(f'\n4. matriz_usuario_item (cruda):')
print(f'   Forma: {matriz_usuario_item.shape}')
print(f'   Para: Analisis y modelos alternativos')

## 6. Guardar archivos procesados

In [ ]:
import os

os.makedirs('../Data/Processed', exist_ok=True)

interaction_matrix.to_csv('../Data/Processed/interaction_matrix.csv')
product_features.to_csv('../Data/Processed/product_features.csv', index=False)
user_features.to_csv('../Data/Processed/user_features.csv', index=False)
matriz_usuario_item.to_csv('../Data/Processed/user_item_df.csv', index=False)

print('Archivos guardados en Data/Processed/:')
print('  - interaction_matrix.csv')
print('  - product_features.csv')
print('  - user_features.csv')
print('  - user_item_df.csv')

---
## 7. Resumen, conclusiones y recomendaciones para el modelado

### Resumen de lo realizado

Se proceso el dataset completo del e-commerce (7 tablas, ~1M de registros) para generar las estructuras necesarias para un sistema de recomendacion:

| Estructura | Tamaño | Destino |
|------------|--------|---------|
| `interaction_matrix` | 19,945 usuarios x 1,197 productos | Collaborative Filtering |
| `product_features` | 1,197 productos x 12 columnas | Content-Based |
| `user_features` | 20,000 usuarios x 11 columnas | Cold Start |
| `matriz_usuario_item` | 529,593 interacciones | Analisis |

### Conclusiones

1. **Esparsidad alta (97.78%)**: la mayoria de los usuarios solo interactuo con pocos productos. Esto confirma que se necesita collaborative filtering (SVD/ALS) para predecir las interacciones faltantes.

2. **Distribucion de scores**: la mayoria de los usuarios tienen scores bajos (page_view), lo que indica que la mayoria de las interacciones son pasivas (solo vistas). Las interacciones activas (add_to_cart, purchase) son la minoria.

3. **Cold start**: 55 clientes (0.27%) no tienen ninguna sesion. Estos requieren recomendaciones basadas en su perfil demografico (edad, pais) y en la popularidad de los productos.

4. **Dataset sintético**: las conversiones son uniformes (~28% en todos los dispositivos, fuentes y paises), lo que limita el poder predictivo de las features demograficas. En un dataset real, estas features serian mucho mas utiles.

### Recomendaciones para el modelado

1. **Collaborative Filtering (SVD/ALS)**: usar `interaction_matrix` como entrada. Es el modelo principal para clientes con historial.

2. **Content-Based**: usar `product_features` para calcular similitud entre productos (cosine similarity). Util para complementar el collaborative filtering.

3. **Hibrido**: combinar collaborative filtering + content-based para cubrir:
   - Clientes nuevos (cold start) -> content-based con features de usuario
   - Productos nuevos -> content-based con features de producto
   - Clientes viejos -> collaborative filtering

4. **Evaluacion**: usar Precision@K y Recall@K para medir la calidad de las recomendaciones. RMSE solo para predecir ratings.

5. **Split temporal**: como las fechas fueron corregidas artificialmente, se recomienda un split aleatorio por usuario (no temporal) para train/test.